# 07 · Writing good tools: names, schemas and errors the model can act on
Level L2 · Part 2 · core · ~20 min · builds on 05

## Goal
Write a tool the model uses correctly: a name, a description that says when to use it, a Pydantic argument schema
that says what a valid value looks like, and an error message that tells the model what to do next. Then watch the
model recover from a bad order number on its own.

## Why it matters
The model never sees your Python function. It sees the **name**, the **description** and the **JSON Schema** you send,
and then the **result** or **error** your code returns. If those are vague, the model guesses. Every guess is a bug
you find later, in production, as a wrong order or a dead-end chat.

Without good tool design, and with it:

| Without good tool design | With it |
|---|---|
| The name says nothing (`lookup`), so the model cannot tell it apart from other tools | The name says what it does (`lookup_order`), and the description says when to use it and when not to |
| A free-text argument (`data`) has no format, so the model guesses what goes in it | The schema names the field and says what it holds: one order number, as the customer wrote it |
| A bad argument raises a stack trace the model cannot read, so the run stops or loops | The error names the bad field and the next step, so the model fixes it and answers |


**What it makes easy:** the model picks the right tool with the right arguments, and recovers from a mistake with no
help from you. **When you don't need it:** a tool your code always calls itself, with one obvious argument, has no
choice for the model to make.

## The idea
A **tool** is an interface written for a model to read. The model reads four things: the **name** (what it does),
the **description** (when to use it, and when not), the **schema** (what each argument means, and which keys are
refused), and the **observation** (what your code returns: a result, or an error with a next step). All four are
part of the prompt, so a good tool is written like good instructions.

Analogy: product labels in a shop. A shelf label that says "Item 7" makes the shopper guess. A label that says
"Trail running shoe, size 9, not for road running" lets them choose, and a note at the counter ("wrong size, try 10")
fixes the mistake without a fuss.

## Picture
```mermaid
flowchart LR
  A[Name + description + schema] --> B[Model writes a call]
  B --> C{Arguments valid?}
  C -- no --> D[Observation: the bad field and the next step]
  D --> B
  C -- yes --> E[Your function runs]
  E --> F[Observation: the result, or an error with a next step]
  F --> B
```
Each loop back to B is a retry. Lesson 06's step cap is the limit that stops a loop that never finishes.

## Step by step
**Step 1 · a vague tool and a good tool (no model call).** Both take an order number. The vague one has a name that says
nothing and one free-text argument. The good one has one argument with a description, and `extra="forbid"` refuses any
other key. Pydantic writes each JSON Schema from the class, so the schema and the check come from one place.

In [1]:
import json
from pydantic import BaseModel, ConfigDict, Field, ValidationError

ORDERS = {"1093": {"status": "in transit", "days_late": 3}, "2207": {"status": "delivered", "days_late": 0}}  # fictional rows


class Anything(BaseModel):  # the vague version: one free-text argument with no format
    data: str


class OrderQuery(BaseModel):  # the good version: one meaning per argument, and no unknown keys
    model_config = ConfigDict(extra="forbid")  # the schema gets additionalProperties: false
    order_id: str = Field(description="The order number from the confirmation email, as the customer wrote it, such as 1093.")


print("vague schema:", json.dumps(Anything.model_json_schema()))
print("good schema: ", json.dumps(OrderQuery.model_json_schema()))

vague schema: {"properties": {"data": {"title": "Data", "type": "string"}}, "required": ["data"], "title": "Anything", "type": "object"}
good schema:  {"additionalProperties": false, "properties": {"order_id": {"description": "The order number from the confirmation email, as the customer wrote it, such as 1093.", "title": "Order Id", "type": "string"}}, "required": ["order_id"], "title": "OrderQuery", "type": "object"}


**Step 2a · the tool and its error.** A tool returns a result, or an error that says what to do next. The error
names the order and the next step, so the model can recover without help from you.

In [2]:
LOOKUP_ORDER_DESCRIPTION = ("Look up one customer order: its status and how many days late it is. Use it when the "
                            "customer gives an order number. Do not use it to search by name, email or date.")


def lookup_order(order_id: str) -> dict:
    if order_id in ORDERS:
        return ORDERS[order_id]
    return {"error": f"no order {order_id} in the system",
            "next_step": "Ask the customer for the number in their confirmation email, then call again with that number. "
                         "Do not guess other numbers."}


TOOLS = {"lookup_order": (lookup_order, OrderQuery)}  # name -> (function, argument schema)
print(lookup_order("1093"))  # the happy path: the row for a known order

{'status': 'in transit', 'days_late': 3}


**Step 2b · the runner.** The runner checks the arguments against the schema before the function runs. When the check
fails, it returns the problem as an observation instead of raising, so the loop keeps going and the model can read the
reason. `.model_dump()` turns the checked Pydantic object back into a plain dict, which the function takes as keyword
arguments. A wrong tool name gets an observation too.

In [3]:
def observation_for_bad_args(err: ValidationError) -> dict:
    problems = [f"{'.'.join(str(p) for p in e['loc'])}: {e['msg']}" for e in err.errors()]
    return {"error": "the arguments are not valid", "problems": problems,
            "next_step": "Fix the named fields and call the tool again."}


def run_tool(name: str, args: dict) -> dict:
    if name not in TOOLS:  # a wrong tool name is an observation too, not a crash
        return {"error": f"no tool named {name}",
                "next_step": f"Call one of these tools: {', '.join(TOOLS)}."}
    func, schema = TOOLS[name]
    try:
        checked = schema.model_validate(args).model_dump()
    except ValidationError as err:
        return observation_for_bad_args(err)  # the model reads this; the loop does not crash
    return func(**checked)


print(run_tool("lookup_order", {"id": "1093"}))        # wrong key: missing field and extra field
print(run_tool("lookup_order", {"order_id": "10930"}))  # right key, unknown order
print(run_tool("refund_order", {"order_id": "1093"}))   # unknown tool name

{'error': 'the arguments are not valid', 'problems': ['order_id: Field required', 'id: Extra inputs are not permitted'], 'next_step': 'Fix the named fields and call the tool again.'}
{'error': 'no order 10930 in the system', 'next_step': 'Ask the customer for the number in their confirmation email, then call again with that number. Do not guess other numbers.'}
{'error': 'no tool named refund_order', 'next_step': 'Call one of these tools: lookup_order.'}


Read the first output: **Field required** points at `order_id`, which is missing. **Extra inputs are not permitted**
comes from `extra="forbid"` in `OrderQuery`, which refuses the `id` key the model sent.

**Step 3 · the clients (the setup).** The key is read from `.env`, never typed here. While a recording replays, a
placeholder is used and no key is sent. `llm_replay` is a course helper: it saves each model call the first time and
replays it later. This notebook makes **four** model calls in total: two to compare the vague and good tools, and two
for the error recovery (the request and the answer after the error).

In [4]:
import sys; sys.path.insert(0, "..")
import llm_replay  # course only: records and replays model calls (not part of the SDK)
llm_replay.start("07_writing-good-tools")

from google import genai  # the official Google SDK; in your project: genai.Client(api_key=...)
from google.genai import types
from tut_env import env  # course only: reads settings from the .env file (not part of the SDK)

NO_KEY = "replay"  # course only: while replaying, the recording answers and this placeholder is never sent
REPLAYING = llm_replay.is_replaying()
gemini = genai.Client(vertexai=True, api_key=NO_KEY if REPLAYING else env("GOOGLE_CLOUD_API_KEY"),
                      http_options=types.HttpOptions(timeout=30_000))
MODEL_GEMINI = "gemini-3.1-flash-lite"
PROMPT = "Where is my parcel for order 1093? It was due on Monday."
print("client ready · replaying:", REPLAYING)

LLM mode: replay (07_writing-good-tools.yaml)


client ready · replaying: True


**Step 4 · the vague tool, then the good tool, same question.** Each tool is declared with its name, description and
schema. Each call prints what the model asked for: the tool it picked and the arguments it sent. Long text is cut short.

In [5]:
def config_for(name: str, description: str, schema) -> types.GenerateContentConfig:
    decl = types.FunctionDeclaration(name=name, description=description, parameters_json_schema=schema.model_json_schema())
    return types.GenerateContentConfig(tools=[types.Tool(function_declarations=[decl])],
                                       tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode="AUTO")),
                                       temperature=0)


def requests_in(response) -> list:
    return [(p.function_call.name, dict(p.function_call.args)) for p in response.candidates[0].content.parts if p.function_call]


def short(calls: list) -> list:
    return [(name, {k: (v[:90] + "…" if isinstance(v, str) and len(v) > 90 else v) for k, v in args.items()})
            for name, args in calls] or "no tool request (plain text reply)"


vague = config_for("lookup", "Lookup.", Anything)
r1 = gemini.models.generate_content(model=MODEL_GEMINI, contents=PROMPT, config=vague)  # call 1
print("vague tool, model asks:", short(requests_in(r1)))

good = config_for("lookup_order", LOOKUP_ORDER_DESCRIPTION, OrderQuery)
r2 = gemini.models.generate_content(model=MODEL_GEMINI, contents=PROMPT, config=good)  # call 2
print("good tool, model asks: ", short(requests_in(r2)))

vague tool, model asks: [('lookup', {'data': 'I am an AI assistant and do not have access to your personal order history, private accoun…'})]
good tool, model asks:  [('lookup_order', {'order_id': '1093'})]


Read the two printed requests. With the vague tool, the model had no format to follow: it called `lookup` and put a
long paragraph of generic advice, starting "I am an AI assistant and do not have access...", into `data`. The model was
given nothing that says what `data` holds, so it filled the field with its own reply. The schema accepted it, so nothing
caught the mistake. With the good tool, the request is `lookup_order` with `order_id: "1093"`, the one value the
description asked for. The model's reply is only a request; your runner decides what happens next.

**Step 5 · recovery from an error (two calls).** The customer's order number is wrong (`10930`, one digit too many).
The model asks for it anyway. Your runner returns the observation from Step 2b: the error names the order and gives a
next step. In the recorded run, the model's second answer asks the customer to check the confirmation email, the next
step the error gave. Both model turns are kept in the history, as in lesson 05.

In [6]:
BAD_PROMPT = "My parcel for order 10930 never came. Where is it?"
history = [types.Content(role="user", parts=[types.Part(text=BAD_PROMPT)])]
r3 = gemini.models.generate_content(model=MODEL_GEMINI, contents=history, config=good)  # call 3: the request
calls = requests_in(r3)
print("model asks:", calls)
history.append(r3.candidates[0].content)  # the model's own turn, unchanged
observations = [run_tool(name, args) for name, args in calls]  # run each tool once; the history and the print share it
history.append(types.Content(role="user", parts=[
    types.Part.from_function_response(name=name, response=obs) for (name, _), obs in zip(calls, observations)]))
print("observation the model reads:", observations[0])

r4 = gemini.models.generate_content(model=MODEL_GEMINI, contents=history, config=good)  # call 4: the answer
print("model answer:", r4.text)
llm_replay.stop()  # the last model call of this notebook; nothing below calls a model

model asks: [('lookup_order', {'order_id': '10930'})]
observation the model reads: {'error': 'no order 10930 in the system', 'next_step': 'Ask the customer for the number in their confirmation email, then call again with that number. Do not guess other numbers.'}
model answer: I'm sorry to hear that your parcel hasn't arrived. I've checked our system for order 10930, but I couldn't find an order with that number.

Could you please double-check the order number in your confirmation email and let me know if it's correct? Once you have the right number, I'll be happy to look into it for you.


## At work
- Teams write each tool description the way they would brief a colleague, then test it: a golden set checks that the
  model picks the right tool with the right arguments, not just that the code runs → lesson 08 (first eval) and lesson 22
  (tool correctness in eval depth).
- Few tools with clear, namespaced names beat a long list of similar ones, because the model picks worse from a crowded
  list → lesson 21 (MCP, where the same names and schemas are published to every client).
- Production loops return validation and business errors as results with a next step, and cap the retries. A crash
  ends the run → lesson 06 (step caps and loop detection) and lesson 32 (reliability). The cap is the one from lesson 06:
  the step limit that stops a loop that never finishes.

## Common mistakes
1. **Letting a validation error escape.** A raw `ValidationError` ends the run with a traceback that the model never
   sees. Return it as an observation instead, as `run_tool` does. A wrong tool name also needs an observation: `run_tool`
   returns one for an unknown name, too.

In [7]:
OrderQuery.model_validate({"id": "1093"})  # the model sent "id", the schema says "order_id"

ValidationError: 2 validation errors for OrderQuery
order_id
  Field required [type=missing, input_value={'id': '1093'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
id
  Extra inputs are not permitted [type=extra_forbidden, input_value='1093', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/extra_forbidden

The fix is the `try` in `run_tool`. The error now reaches the model as a result it can read, with the field names.

2. **Vague names and one-word descriptions.** `lookup` and `Lookup.` tell the model nothing about when to pick the
   tool. Write the description as you would brief a new colleague: what it does, when to use it, when not to.
3. **Free-text arguments.** A `data: str` argument invites any format. Name the field for one meaning, and say the
   format in its description.

## Try it
Write `not_found(order_id)` for an order that does not exist. It must return an observation that names the number the
model sent, says what a valid number looks like, and gives one next step. No model call is needed.

In [8]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [9]:
def not_found(order_id: str) -> dict:
    return {"error": f"no order {order_id} in the system",
            "expected": "the order number from the confirmation email, digits only, such as 1093",
            "next_step": "Ask the customer for the number in their confirmation email, then call again."}


print(not_found("10930"))

{'error': 'no order 10930 in the system', 'expected': 'the order number from the confirmation email, digits only, such as 1093', 'next_step': 'Ask the customer for the number in their confirmation email, then call again.'}


## Recap
- The model sees only the name, description, schema and observation: write all four for a reader that cannot ask you a question.
- A schema with one meaning per argument, and extra keys refused, makes a wrong call easy to catch before your function runs.
- Return an error as an observation that names the bad field and gives the next step; a raw exception ends the run.
- In the recorded run, the model read the error's next step and asked the customer to check the confirmation email.

**Interview:** "A model keeps calling your tool with the wrong arguments. What do you change?" Strong answer: "The name,
description and schema are its only instructions, so I fix those first: one meaning per argument, extra keys refused,
and a description of when to use the tool. Then every error returns the bad field and the next step, so it can recover."